# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmanDbz1101/FlyRank-/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the validated model output into a content action playbook.
It includes ranked actions with reason codes, archetype-to-action mapping,
intended use, limits, human-review rules, and monitoring triggers.
The queue CSV exported here becomes the recommendations section of the research paper.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, sys, json
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ".."))

from capstone.src.data.warehouse import load_population
from capstone.src.data.features import build_feature_vector, FEATURES8, get_feature_matrix
from capstone.src.models.baseline import apply_baseline
from capstone.src.models.classifier import train_and_evaluate
from capstone.src.evaluation.metrics import precision_at_k

OUT = "../outputs"
os.makedirs(OUT, exist_ok=True)
FIG = "../figures"
os.makedirs(FIG, exist_ok=True)

In [ ]:
# Load data and train model
pop = load_population()
pop = build_feature_vector(pop)
pop = apply_baseline(pop)
results = train_and_evaluate(pop)
pop["lr_score"] = results["lr"]
base_rate = pop.is_declining.mean()
print(f"Population: {len(pop):,} pages, base rate: {base_rate:.4f}")

## 1. Ranked actions + reason codes

The top-50 pages by LR score, with actions and reason codes.

In [ ]:
# Build the action queue
top50 = pop.nlargest(50, "lr_score").copy()
top50["rank"] = range(1, len(top50) + 1)

# Assign reason codes based on features
def assign_reason_code(row):
    codes = []
    if row.get("ctr_mar", 0) < 1.0 and row.get("mar_avg_position", 999) < 10:
        codes.append("CTR_GAP_TOP10")
    if row.get("mar_clicks", 0) == 0 and row.get("mar_avg_position", 999) < 20:
        codes.append("ZERO_CLICKS")
    if row.get("has_feb_data", 0) == 0:
        codes.append("NO_FEB_DATA")
    m = row.get("momentum_feb_to_mar_pct")
    if pd.notna(m) and m > 200:
        codes.append("SPIKE_FADING")
    if pd.notna(m) and m < -20:
        codes.append("STALE_CONTENT")
    return ", ".join(codes) if codes else "GENERAL_DECLINE"

top50["reason_code"] = top50.apply(assign_reason_code, axis=1)

# Assign actions
def assign_action(row):
    codes = row["reason_code"]
    if "CTR_GAP_TOP10" in codes or "ZERO_CLICKS" in codes:
        return "REFRESH"
    if "SPIKE_FADING" in codes or "STALE_CONTENT" in codes:
        return "MONITOR"
    if "NO_FEB_DATA" in codes:
        return "MONITOR"
    return "REFRESH"

top50["action"] = top50.apply(assign_action, axis=1)

# Assign confidence
def assign_confidence(row):
    codes = row["reason_code"].split(", ")
    if len(codes) >= 3:
        return "high"
    if len(codes) >= 2:
        return "medium"
    return "low"

top50["confidence"] = top50.apply(assign_confidence, axis=1)

# Human-readable explanation
def explain(row):
    parts = []
    pos = row.get("mar_avg_position", 0)
    ctr = row.get("ctr_mar", 0)
    imp = row.get("mar_impressions", 0)
    m = row.get("momentum_feb_to_mar_pct")
    if pd.notna(pos) and pos < 10:
        parts.append(f"ranks at position {pos:.1f}")
    if ctr < 1.0:
        parts.append(f"CTR is {ctr:.2f}%")
    parts.append(f"{imp:,.0f} March impressions")
    if pd.notna(m):
        parts.append(f"momentum {m:+.0f}%")
    return "; ".join(parts)

top50["explanation"] = top50.apply(explain, axis=1)

# Display
display_cols = ["rank", "content_type", "mar_impressions", "ctr_mar", "lr_score", "action", "reason_code", "confidence"]
print("=== Top 50 Action Queue ===")
print(top50[display_cols].to_string(index=False))
print()
print(f"Actions: {top50.action.value_counts().to_dict()}")
print(f"Confidence: {top50.confidence.value_counts().to_dict()}")

In [ ]:
# Archetype-to-action mapping
print("=== Archetype → Action Mapping ===")
print()
archetypes = pd.DataFrame({
    "Archetype": [
        "High-impression, zero clicks, top-10",
        "High-impression, low CTR, top-20",
        "No February data, zero clicks",
        "Spike fading (>200% MoM decline)",
        "Stale content (>1 year, declining)",
        "Moderate decline (20-50% MoM)",
    ],
    "Action": ["REFRESH", "REFRESH", "MONITOR", "MONITOR", "REFRESH", "MONITOR"],
    "Priority": ["critical", "high", "medium", "medium", "high", "low"],
    "What to check": [
        "Snippet, title tag, meta description, schema markup",
        "Snippet appeal, query intent match, SERP features",
        "Cannot assess momentum — watch for another period",
        "May be trending query fading; wait for stabilization",
        "Content freshness, internal linking, indexing status",
        "Monitor for another period before acting",
    ],
})
print(archetypes.to_string(index=False))

## 2. Intended use and limits

**Intended use:** Content editors with limited review time use this playbook to triage which pages to investigate first. The model ranks pages by decline risk; the reason codes suggest where to look; the action recommendation tells the editor what to check.

**Who uses this:**
- Content editors reviewing monthly performance
- SEO leads prioritizing refresh efforts
- Product managers assessing content health

**For what:**
- Triage: which 50 pages out of 125K to review this month
- Direction: what to check on each page (snippet, content freshness, internal links)
- Priority: which pages are critical vs. medium vs. low

**Where it stops being valid:**
- The model is trained on one portfolio (44 clients, March→April 2026). It may not generalize to other industries or time periods.
- The model uses observational data. It cannot prove that refreshing a page will recover traffic.
- The model does not know about SERP features (featured snippets, knowledge panels), competitor changes, or Google algorithm updates.
- The model does not account for seasonality. A page declining in April may recover in May.
- The model does not distinguish between intentional content pruning and unintentional decline.

In [ ]:
# Summary of limits
print("=== Intended Use and Limits ===")
print()
print("INTENDED USE:")
print("  - Triage which pages to review first (not which to fix)")
print("  - Suggest where to look (snippet, content, links)")
print("  - Prioritize by confidence and impact")
print()
print("LIMITS:")
print("  - One portfolio, one time window (March→April 2026)")
print("  - Observational data — no causal claims")
print("  - No SERP feature awareness")
print("  - No competitor data")
print("  - No seasonality adjustment")
print("  - Cannot distinguish intentional pruning from decline")
print("  - 51% of rows lack GA4 data — engagement signals are partial")

## 3. Human review + the no-go list

**What a person must check before acting:**
1. **Actual page content** — Does the page still exist? Is the content accurate? Has it been updated since March?
2. **Current SERP layout** — Are there new SERP features (featured snippets, knowledge panels) that changed the click landscape?
3. **Competitor changes** — Did a competitor publish a better page? Did the competitive landscape shift?
4. **Seasonal patterns** — Is this a seasonal dip? Does the page historically recover?
5. **Algorithm updates** — Did Google roll out a core update in April? (Check Google Search Status Dashboard.)
6. **Business context** — Is this page strategically important? Does the business want it to rank?

**What should never be automated:**
1. **Rewriting content** — The model identifies which pages to review; a human decides what to change.
2. **Removing or unpublishing pages** — Decline risk does not mean the page should be removed.
3. **Penalty decisions** — The model cannot distinguish algorithmic penalties from natural decline.
4. **Publishing without editor review** — All changes must go through an editorial workflow.
5. **Budget allocation** — The model suggests priorities; a human decides how to spend the budget.

In [ ]:
print("=== Human Review Requirements ===")
print()
print("BEFORE acting on any REFRESH recommendation, a human must:")
print("  1. Open the actual page and verify it exists and is accurate")
print("  2. Check the current SERP for the target query")
print("  3. Look at competitor pages for the same query")
print("  4. Check Google Search Status Dashboard for recent updates")
print("  5. Confirm the page is strategically important to the business")
print()
print("WHAT SHOULD NEVER BE AUTOMATED:")
print("  ✗ Rewriting content without editor review")
print("  ✗ Removing or unpublishing pages")
print("  ✗ Making penalty or demotion decisions")
print("  ✗ Publishing changes without approval")
print("  ✗ Allocating budget based solely on model scores")

## 4. Monitoring / retrain triggers

**What would make the recommendations go stale:**
- **Google core update**: If Google rolls out a major algorithm update, the model's learned patterns may no longer apply. Re-score immediately after the update stabilizes.
- **Data drift**: If the distribution of features shifts significantly (e.g. average CTR drops across the portfolio), the model's calibration is off. Monitor feature distributions monthly.
- **Composition change**: If the content portfolio changes >20% in size or composition (new client, major content purge), retrain.
- **Performance drop**: If P@50 on new data drops below 0.60, the model is no longer useful. Retrain.
- **Seasonal shift**: If the content has strong seasonal patterns, the model may not generalize across seasons.

**Proposed cadence:**
- **Monthly**: Re-score all pages with the current model. Flag new declines.
- **Quarterly**: Retrain the model on the latest data. Compare P@50 to previous quarter.
- **On demand**: After any Google core update, re-score within 2 weeks.

In [ ]:
print("=== Monitoring / Retrain Triggers ===")
print()
print("RETRAIN IF:")
print("  1. Google rolls out a core algorithm update")
print("  2. P@50 on new data drops below 0.60")
print("  3. Feature distributions shift > 1 std dev from training data")
print("  4. Content portfolio changes > 20% in size or composition")
print()
print("PROPOSED CADENCE:")
print("  - Monthly:   Re-score all pages with current model")
print("  - Quarterly: Retrain model, compare P@50 to previous quarter")
print("  - On demand: Re-score within 2 weeks of any core update")
print()
print("COST / VALUE THINKING:")
print("  - Model training: ~5 min (8 features, 125K rows, LR) — low cost")
print("  - Agent pipeline: ~3 sec per page — feasible for top 50")
print("  - Editor review: ~5 min per page — the real bottleneck")
print("  - For a 50-page budget: ~4 hours of editor time")
print("  - Value: 44 of 50 pages are actually declining (P@50 = 0.880)")
print("  - vs. baseline: 35 of 50 pages (P@50 = 0.700)")
print("  - Improvement: 9 more real declines caught per 50-page review cycle")

## 5. Exports for the paper

Write the queue CSV and summary JSON to `work/outputs/` for the paper to build on.

In [ ]:
# Export the action queue
export_cols = ["rank", "content_hash_id", "content_type", "mar_impressions", "ctr_mar",
               "lr_score", "action", "reason_code", "confidence", "explanation"]
queue_path = f"{OUT}/action_playbook_queue.csv"
top50[export_cols].to_csv(queue_path, index=False)
print(f"Exported: {queue_path} ({len(top50)} rows)")

# Export summary JSON
summary = {
    "queue_size": 50,
    "population": len(pop),
    "base_rate": float(base_rate),
    "model": "Logistic Regression",
    "validation": "Client-grouped 4-fold GroupKFold",
    "precision_at_50": float(precision_at_k(pop.is_declining.values, pop.lr_score.values, 50)),
    "actions": top50.action.value_counts().to_dict(),
    "reason_codes": top50.reason_code.value_counts().to_dict(),
    "archetype_mapping": {
        "CTR_GAP_TOP10": "REFRESH",
        "ZERO_CLICKS": "REFRESH",
        "NO_FEB_DATA": "MONITOR",
        "SPIKE_FADING": "MONITOR",
        "STALE_CONTENT": "REFRESH",
        "GENERAL_DECLINE": "REFRESH",
    },
    "retrain_triggers": [
        "Google core update",
        "P@50 drops below 0.60",
        "Feature drift > 1 std dev",
        "Portfolio changes > 20%",
    ],
    "no_go_list": [
        "Rewriting content without editor review",
        "Removing or unpublishing pages",
        "Penalty decisions",
        "Publishing without approval",
        "Budget allocation from model scores alone",
    ],
}
summary_path = f"{OUT}/action_playbook_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Exported: {summary_path}")

In [ ]:
# Verify exports
print("=== Export Verification ===")
print()
q = pd.read_csv(queue_path)
print(f"Queue CSV: {len(q)} rows, columns: {list(q.columns)}")
print(f"Actions: {q.action.value_counts().to_dict()}")
print()
with open(summary_path) as f:
    s = json.load(f)
print(f"Summary JSON: {list(s.keys())}")
print(f"Precision@50: {s['precision_at_50']:.4f}")
print(f"Base rate: {s['base_rate']:.4f}")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.